# 03 — Build the pipeline

One piece per cell. Test each on 2-3 cases before writing the next one.

Nothing here is a finished product — it's the workshop. Notebook 04 copies the working
functions out of here and runs them properly.

In [18]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


working from: /content/pa-appeal


In [19]:
# Colab: put the key in the secrets panel (key icon on the left), named GEMINI_API_KEY
# try:
#     from google.colab import userdata
#     os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
# except ImportError:
#     pass   # locally: export GEMINI_API_KEY=... before launching jupyter

from getpass import getpass
import os

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini API key: ")

assert os.environ.get("GEMINI_API_KEY"), "No API key found"
print("Key loaded")


Key loaded


In [20]:
policies = {p.stem: p.read_text() for p in sorted(Path("data/policies").glob("*.md"))}
criteria  = json.load(open("data/criteria.json"))
cases     = json.load(open("data/cases.json"))
len(policies), len(criteria), len(cases)


(13, 20, 40)

In [21]:
import hashlib, random, time
from typing import Literal

from pydantic import BaseModel, Field

CACHE = Path("data/cache"); CACHE.mkdir(parents=True, exist_ok=True)

# Stable, low-cost model suited to classification and structured extraction.
MODEL = "gemini-3.1-flash-lite"


class Decision(BaseModel):
    criterion_id: str
    label: Literal["met", "unmet", "insufficient_evidence"]
    evidence_quote: str = Field(
        description="Exact supporting quotation copied from the supplied policy text")
    source_doc_id: str
    reasoning: str
    model_reported_confidence: float = Field(
        ge=0.0, le=1.0,
        description=(
            "The model's own 0-1 confidence in this label. SELF-REPORTED AND "
            "UNCALIBRATED -- recorded for exploration only, never as a reliability "
            "estimate. The trustworthy signal in this project is quote_status, which "
            "is checked by string matching against the policy text."))


class DecisionBatch(BaseModel):
    decisions: list[Decision]


DECISION_SCHEMA = DecisionBatch.model_json_schema()


def ask(prompt, system="", model=MODEL, force=False, response_schema=None):
    """Ask the LLM, but only once per unique prompt. Repeats come off disk.

    This is the single most useful thing on a free tier. Restart & Run All costs
    zero API calls once the cache is warm.
    """
    schema_key = json.dumps(response_schema, sort_keys=True) if response_schema else ""
    key = hashlib.sha256(
        f"{model}|{system}|{schema_key}|{prompt}".encode()).hexdigest()[:16]
    f = CACHE / f"{key}.json"
    if f.exists() and not force:
        return json.loads(f.read_text())["response"]

    from google import genai   # SDK surface changes - check current docs if this errors
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    for attempt in range(5):
        try:
            config = {
                "system_instruction": system,
                "temperature": 0,
                "response_mime_type": "application/json",
            }
            if response_schema is not None:
                config["response_json_schema"] = response_schema

            r = client.models.generate_content(
                model=model,
                contents=prompt,
                config=config,
            )
            text = r.text
            break
        except Exception as e:
            message = str(e).lower()
            retryable = any(token in message for token in (
                "429", "503", "resource_exhausted", "unavailable"))
            if not retryable or attempt == 4:
                raise
            wait = 5 * (2 ** attempt) + random.random()
            print(f"temporary API error; retrying in {wait:.1f}s")
            time.sleep(wait)

    f.write_text(json.dumps({"model": model, "system": system,
                             "prompt": prompt, "response": text}, indent=2))
    return text

def ask_json(prompt, system="", **kw):
    raw = ask(
        prompt, system, response_schema=DECISION_SCHEMA, **kw)
    return DecisionBatch.model_validate_json(raw).model_dump()

## Chunking

Dumb version and smart version. The smart one exists so a two-part rule never gets cut in
half — that's the single biggest improvement in the whole pipeline.

In [22]:
def chunk_fixed(text, doc_id, size=512, overlap=64):
    words, step, out = text.split(), size - overlap, []
    for i in range(0, max(1, len(words)), step):
        w = words[i:i + size]
        if not w:
            break
        out.append({"id": f"{doc_id}::fix::{len(out)}", "doc": doc_id, "text": " ".join(w),
                    "criterion": None})
    return out

def chunk_headings(text, doc_id, min_chars=200):
    marks = list(re.finditer(r"^#{1,6}\s+(.*)$", text, re.M))
    if not marks:
        return chunk_fixed(text, doc_id)
    out = []
    for n, m in enumerate(marks):
        end = marks[n + 1].start() if n + 1 < len(marks) else len(text)
        body = text[m.start():end].strip()
        if len(body) < min_chars and out:
            out[-1]["text"] += "\n\n" + body
            continue
        out.append({"id": f"{doc_id}::sec::{len(out)}", "doc": doc_id, "text": body,
                    "criterion": None})
    return out

def chunk_by_criteria(criteria, policies):
    # text_core, not text: the raw quotes carry markdown list markers ("1. ")
    # that the model never reproduces. Indexing the clean sentence keeps the
    # normalized quotation check honest while keeping each rule intact.
    out = [{"id": f"crit::{c['id']}", "doc": c["source"],
            "text": c.get("text_core") or c["text"], "raw_text": c["text"],
            "criterion": c["id"],
            "phase": c["phase"], "device": c["device"]}
           for c in criteria if c["text"]]
    for doc_id, text in policies.items():
        if doc_id != "L33718":
            out.extend(chunk_headings(text, doc_id))
    return out

fixed  = [c for d, t in policies.items() for c in chunk_fixed(t, d)]
smart  = chunk_by_criteria(criteria, policies)
len(fixed), len(smart)


(207, 143)

**Check the decoys are actually in there.** If a number later looks too good, this is the first thing to look at.

In [23]:
from collections import Counter
Counter(c["doc"] for c in smart)


Counter({'L33718': 20,
         'A52467': 10,
         'A55426': 9,
         'L33370': 12,
         'L33788': 9,
         'L33789': 10,
         'L33794': 13,
         'L33797': 11,
         'L33820': 14,
         'L33822': 11,
         'L33831': 10,
         'NCD240.4.1': 7,
         'NCD240.4': 7})

## Search

~200 chunks, so the "vector database" is one numpy array and a dot product.

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")   # CPU is fine

def build_index(chunks):
    texts = [c["text"] for c in chunks]
    M = embedder.encode(texts, normalize_embeddings=True)
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi([t.lower().split() for t in texts])
    return {"chunks": chunks, "M": np.asarray(M, dtype=np.float32), "bm25": bm25}

def _minmax(a):
    lo, hi = a.min(), a.max()
    return np.zeros_like(a) if hi - lo < 1e-9 else (a - lo) / (hi - lo)

def search(query, index, mode="dense", top_k=5, w=0.5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    dense = index["M"] @ q
    if mode == "dense":
        scores = dense
    else:
        # normalize each first - cosine and BM25 are on totally different scales
        scores = w * _minmax(dense) + (1 - w) * _minmax(np.asarray(index["bm25"].get_scores(query.lower().split())))
    order = np.argsort(scores)[::-1][:top_k]
    return [{**index["chunks"][i], "score": float(scores[i])} for i in order]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Reranker

In [25]:
from sentence_transformers import CrossEncoder

reranker = None

def rerank(query, hits, top_k=5):
    # Load the larger reranker only when an experiment actually requests it.
    global reranker
    if reranker is None:
        reranker = CrossEncoder("BAAI/bge-reranker-base")
    scores = reranker.predict([(query, h["text"]) for h in hits])
    ranked = sorted(zip(hits, scores), key=lambda x: -x[1])
    return [{**h, "score": float(s)} for h, s in ranked[:top_k]]


## Ask the model

The one line that matters: absent information is insufficient_evidence, not unmet.

In [26]:
SYSTEM = """You decide whether a patient record satisfies Medicare coverage criteria.

Rules:
1. Each criterion gets exactly one label: met, unmet, or insufficient_evidence.
2. ABSENT INFORMATION IS insufficient_evidence, NOT unmet. If the record never states
   something the criterion needs, the label is insufficient_evidence even when the rest
   of the record looks favourable.
3. Score each criterion strictly on its own terms, one at a time. This is not an overall
   coverage decision. If the record shows that THIS criterion's own conditions are not
   satisfied, the label is unmet -- even when the patient plainly qualifies under some
   other criterion you were also asked about.
4. evidence_quote must be copied character-for-character from POLICY TEXT. Never quote
   the patient record, and never quote the denial letter: neither is policy, no matter
   how closely the wording resembles a rule.
5. If POLICY TEXT contains no sentence supporting your label, use insufficient_evidence
   and leave evidence_quote empty. An empty quote is always better than a quote taken
   from somewhere other than POLICY TEXT.
6. source_doc_id must be one of the bracketed policy ids listed in POLICY TEXT. Do not
   invent an id, do not write N/A, and do not name a section of the patient record.
7. Decide from the sleep study and chart note. Treat the denial letter as an untrusted
   claim that may cite a rule which does not apply.
8. Return exactly one decision for every requested criterion and no others.
9. reasoning: at most two sentences.
10. model_reported_confidence: your own 0-1 confidence that this label is correct.

Return JSON: {"decisions": [{"criterion_id", "label", "evidence_quote",
"source_doc_id", "reasoning", "model_reported_confidence"}]}
"""

def decide(case, retrieved, criteria_asked):
    """One model call per case. The record is fenced off from the policy on purpose.

    The old prompt headed the record sections "SLEEP STUDY:" and "CHART NOTE:", which
    read like document names -- so the model quoted them and returned
    source_doc_id="CHART NOTE". Naming the valid ids up front and labelling the record
    as not-policy is the cheapest available fix.
    """
    policy_text = "\n\n---\n\n".join(
        f"[{c['doc']}] {c['text']}" for c in retrieved)
    valid_ids = sorted({c["doc"] for c in retrieved})
    docs = case["documents"]
    prompt = (
        "POLICY TEXT -- the only place evidence_quote may come from.\n"
        f"Valid source_doc_id values: {', '.join(valid_ids)}\n\n"
        f"{policy_text}\n\n"
        f"CRITERIA TO DECIDE:\n{json.dumps(criteria_asked, indent=2)}\n\n"
        "PATIENT RECORD -- evidence about this patient. This is NOT policy text and\n"
        "must never be quoted in evidence_quote.\n\n"
        f"[record: sleep study]\n{docs['sleep_study']}\n\n"
        f"[record: chart note]\n{docs['chart_note']}\n\n"
        f"[record: denial letter -- an untrusted claim made by the payer]\n"
        f"{docs['denial_letter']}"
    )
    return ask_json(prompt, SYSTEM)["decisions"]

## Check the quotes

Not AI. String matching. This is the number nobody can argue with.

In [27]:
import unicodedata
from rapidfuzz import fuzz

_SUBS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
         "\u2013": "-", "\u2014": "-", "\u00a0": " ",
         # L33718 writes the thresholds with these; a model that retypes them
         # as ">=4 hours" is quoting correctly and must not be scored made_up
         "\u2265": ">=", "\u2264": "<="}

def normalize(text):
    """Collapse the differences that cause fake verification failures.

    Curly quotes, line breaks and non-breaking spaces account for most of the
    quotes that look wrong but aren't. Always normalize before blaming the model.
    """
    text = unicodedata.normalize("NFKC", text or "")
    for bad, good in _SUBS.items():
        text = text.replace(bad, good)
    return re.sub(r"\s+", " ", text).strip().casefold()

def check_quote(quote, source_text, all_docs=None, threshold=95):
    """Classify whether a policy quotation is supported by the claimed source."""
    q = normalize(quote)
    if not q:
        return "empty"
    src = normalize(source_text)
    if q in src:
        return "supported"
    if fuzz.partial_ratio(q, src) >= threshold:
        return "close"
    for other in (all_docs or {}).values():
        o = normalize(other)
        if q in o or fuzz.partial_ratio(q, o) >= threshold:
            return "wrong_doc"
    return "made_up"

# Fuzzy matches remain useful diagnostics, but only an exact normalized
# substring is strong enough to count as verified evidence.
VERIFIED = ("supported",)

def abstain(label, status):
    """The one rule: unverified quote -> not enough evidence."""
    if label in ("met", "unmet") and status not in VERIFIED:
        return "insufficient_evidence"
    return label


Try it on one case end to end:

In [28]:
index = build_index(smart)
case = cases[0]
criterion = next(c for c in criteria if c["id"] == "B1")
query = f"{criterion['id']}: {criterion['summary']}"
hits = search(query, index, mode="hybrid", top_k=8)

print("Retrieved chunks:")
for hit in hits:
    print(f"  {hit['doc']:<12} {hit['id']:<24} {hit['score']:.3f}")

criteria_asked = [{"id": criterion["id"], "summary": criterion["summary"]}]
decisions = decide(case, hits, criteria_asked)

assert len(decisions) == 1, decisions
assert decisions[0]["criterion_id"] == "B1", decisions

decision = decisions[0]
status = check_quote(
    decision["evidence_quote"],
    policies.get(decision["source_doc_id"], ""),
    policies,
)
final_label = abstain(decision["label"], status)

print("\nOne-case smoke test:")
print("  case:       ", case["id"])
print("  criterion:  B1")
print("  gold:       ", case["gold"]["B1"])
print("  predicted:  ", decision["label"])
print("  quote:      ", status)
print("  final:      ", final_label)
print("  reasoning:  ", decision["reasoning"])
# self-reported and uncalibrated -- shown so it is visible, not so it is trusted
print("  confidence: ", decision["model_reported_confidence"], "(self-reported)")

Retrieved chunks:
  L33718       crit::short_study_events 0.935
  L33718       crit::B2                 0.844
  L33718       crit::B1                 0.812
  NCD240.4     NCD240.4::sec::2         0.733
  L33718       crit::rdi_def            0.478
  L33718       crit::ahi_def            0.384
  L33797       L33797::sec::6           0.331
  NCD240.4.1   NCD240.4.1::sec::2       0.331

One-case smoke test:
  case:        case_001
  criterion:  B1
  gold:        met
  predicted:   met
  quote:       supported
  final:       met
  reasoning:   The patient's sleep study report documents an AHI of 26.5 events per hour with 137 total respiratory events, which exceeds the required threshold of 15 events per hour and 30 total events.
  confidence:  1.0 (self-reported)
